# 01 · Data, EDA and feature engineering
**Project:** Credit default risk: a calibrated, explainable and fairness-audited ML system
**Data:** [Home Credit Default Risk](https://www.kaggle.com/competitions/home-credit-default-risk) (307k loan applications, 7 relational tables)

This notebook downloads the data, analyses it, builds ~400 applicant-level features from the
relational tables, and creates the **train / holdout split**. The holdout is not touched again until notebook 03.

> Before running: (1) accept the competition rules on Kaggle (Rules tab -> "I Understand and Accept");
> (2) add your Kaggle API token as a **Colab secret** named `KAGGLE_API_TOKEN` (key icon in the left sidebar,
> toggle "Notebook access" on). Runtime: CPU is fine.

In [ ]:
# ---------- Setup: run this cell first ----------
import os, sys, subprocess, json, warnings
warnings.filterwarnings("ignore")
SMOKE = os.environ.get("CR_SMOKE") == "1"          # local smoke test on synthetic data (ignore on Colab)
if SMOKE:
    PROJECT_DIR = os.environ["CR_PROJECT_DIR"]
else:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/masters-projects/credit-risk"   # <- change if you put it elsewhere
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{PROJECT_DIR}/requirements-colab.txt"], check=True)
sys.path.insert(0, f"{PROJECT_DIR}/src")
ART = f"{PROJECT_DIR}/artifacts"; FIG = f"{PROJECT_DIR}/reports/figures"
os.makedirs(ART, exist_ok=True); os.makedirs(FIG, exist_ok=True)

import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
SEED = 42

def savefig(name):
    plt.tight_layout(); plt.savefig(f"{FIG}/{name}.png", dpi=160, bbox_inches="tight"); plt.show()

def save_json(obj, name):
    with open(f"{ART}/{name}.json", "w") as f: json.dump(obj, f, indent=2, default=float)
print("Project dir:", PROJECT_DIR)

## 1. Download and load the data

In [ ]:
from creditrisk import data as D
if SMOKE:
    from creditrisk.synthetic import make_synthetic_tables
    tables = make_synthetic_tables(n=4000)
else:
    print("Kaggle auth:", D.setup_kaggle_credentials(legacy_kaggle_json="/content/drive/MyDrive/masters-projects/kaggle.json"))
    DATA_DIR = D.download_home_credit("/content/data/home-credit")   # local disk is much faster than Drive
    tables = D.load_tables(DATA_DIR)
app = tables["application"]
print(app.shape)

## 2. Target and class imbalance

In [ ]:
rate = app["TARGET"].mean()
print(f"Default rate: {rate:.2%}  ->  {1/rate:.1f} : 1 imbalance")
fig, ax = plt.subplots(figsize=(4, 3))
app["TARGET"].value_counts().sort_index().plot.bar(ax=ax, color=["#4C72B0", "#C44E52"])
ax.set_xticklabels(["repaid (0)", "default (1)"], rotation=0); ax.set_title("Target distribution")
savefig("01_target")

## 3. Data quality: missing values and anomalies

In [ ]:
miss = D.missing_report(app)
print(f"{(miss.missing_share > 0).sum()} / {app.shape[1]} columns have missing values; "
      f"{(miss.missing_share > 0.5).sum()} are >50% missing")
fig, ax = plt.subplots(figsize=(7, 6))
miss.head(30)["missing_share"][::-1].plot.barh(ax=ax, color="#8172B3"); ax.set_xlabel("share missing")
ax.set_title("Top-30 columns by missingness (application table)")
savefig("01_missingness")

In [ ]:
anom = app["DAYS_EMPLOYED"] == 365243
print(f"DAYS_EMPLOYED == 365243 (sentinel ~1000 years): {anom.mean():.2%} of applicants")
print(app.groupby(anom)["TARGET"].mean().rename({False: "normal", True: "sentinel"}))
print("Income type of sentinel rows:\n", app.loc[anom, "NAME_INCOME_TYPE"].value_counts().head(3))
# -> almost all pensioners/unemployed; we set it to NaN and keep a flag (DAYS_EMPLOYED_ANOM)

## 4. What predicts default? Univariate views

In [ ]:
df = app.copy()
df["AGE_YEARS"] = -df["DAYS_BIRTH"] / 365.25
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, col in zip(axes[0], ["EXT_SOURCE_2", "EXT_SOURCE_3"]):
    q = pd.qcut(df[col], 10, duplicates="drop")
    df.groupby(q, observed=True)["TARGET"].mean().plot(ax=ax, marker="o")
    ax.set_title(f"Default rate by {col} decile"); ax.set_xlabel(""); ax.tick_params(axis="x", rotation=60)
df.groupby(pd.cut(df["AGE_YEARS"], [20, 25, 30, 35, 40, 45, 50, 55, 60, 70]), observed=True)["TARGET"].mean() \
  .plot.bar(ax=axes[1, 0], color="#55A868"); axes[1, 0].set_title("Default rate by age")
df.groupby("NAME_EDUCATION_TYPE", observed=True)["TARGET"].mean().sort_values() \
  .plot.barh(ax=axes[1, 1], color="#DD8452"); axes[1, 1].set_title("Default rate by education"); axes[1, 1].set_ylabel("")
savefig("01_univariate")

In [ ]:
# Sensitive attributes we will audit later (gender is NOT used as a model feature)
print(app.groupby("CODE_GENDER", observed=True)["TARGET"].agg(["mean", "size"]))

In [ ]:
# Strongest linear correlations with the target (numeric columns)
num = app.select_dtypes("number").drop(columns=["SK_ID_CURR"])
corr = num.corr()["TARGET"].drop("TARGET").sort_values()
pd.concat([corr.head(8), corr.tail(8)]).to_frame("pearson_r")

## 5. Feature engineering
One row per applicant. From each auxiliary table we aggregate the applicant's *history*:
* **bureau / bureau_balance**: loans at other lenders, active debt, overdue amounts, monthly delinquency status
* **previous_application**: past Home Credit applications, refusal rate, requested vs. granted amounts
* **installments_payments**: days-past-due, under-payment, recent (12-month) behaviour
* **POS_CASH / credit_card_balance**: delinquency, utilisation, drawings
* **domain ratios**: credit/income, annuity/income, annuity/credit (≈ loan term), external-score combinations

All aggregations are per applicant and use only data from before the application, so the joins cannot leak the target.

In [ ]:
from creditrisk.features import build_features, feature_columns
feat = build_features(tables)
FEATURES = feature_columns(feat)                  # excludes ID, TARGET and CODE_GENDER
print(f"Feature matrix: {feat.shape},  model features: {len(FEATURES)}")
print(f"Memory: {feat.memory_usage(deep=True).sum()/1e6:.0f} MB")

## 6. Train / holdout split (stratified 80/20; the holdout is used once, in notebook 03)

In [ ]:
from sklearn.model_selection import train_test_split
tr_ids, ho_ids = train_test_split(feat["SK_ID_CURR"], test_size=0.2, stratify=feat["TARGET"], random_state=SEED)
feat["SPLIT"] = np.where(feat["SK_ID_CURR"].isin(ho_ids), "holdout", "train")
print(feat.groupby("SPLIT")["TARGET"].agg(["size", "mean"]))
feat.to_parquet(f"{ART}/features.parquet", index=False)
save_json({"features": FEATURES}, "feature_list")
save_json({"n_rows": int(len(feat)), "n_features": len(FEATURES), "default_rate": float(feat.TARGET.mean()),
           "n_train": int((feat.SPLIT == "train").sum()), "n_holdout": int((feat.SPLIT == "holdout").sum()),
           "days_employed_sentinel_share": float(anom.mean()),
           "n_cols_over_50pct_missing": int((miss.missing_share > 0.5).sum())}, "eda_summary")
print("Saved features.parquet to", ART)

## Observations (fill in after running, in your own words)
* Imbalance: ...
* Strongest signals: ...
* Data-quality issues and how they were handled: ...